# Evaluation Lab: Diagnosing and Fixing Post-Training Failures

In this lab, you will investigate two models that were fine-tuned on subtly corrupted data.  
Your goal is to **discover** the failure modes, **build verifiers** to detect bad training samples, and **retrain** the models on cleaned data.

Two models, two failure modes:
1. **Code model** -- fine-tuned on coding problems where ~20% of training solutions contain subtle bugs.
2. **QA model** -- fine-tuned on trivia questions where ~20% of training answers are plausible but wrong.

You will work through five sections:
- **Section 0**: Setup and loading
- **Section 1**: Initial evaluation -- how do the models perform?
- **Section 2**: Training data investigation -- what went wrong?
- **Section 3**: Build a verifier -- can you catch the bad data?
- **Section 4**: Filter and retrain -- can you fix the models?
- **Section 5**: Reflection

**This is the solved version with all implementations filled in.**

---
## Section 0: Setup and Load

This section loads the pre-trained artifacts (models, data, manifest) that were created by the teacher prep notebook.  
Nothing to implement here -- just run these cells and verify everything loads correctly.

In [19]:
# =============================================================
# Configuration -- update these paths for your environment
# =============================================================
from pathlib import Path

ARTIFACTS_DIR = Path("./lab_artifacts")
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

DATA_DIR = ARTIFACTS_DIR / "data"
CODE_TRAIN_PATH = DATA_DIR / "code_train.json"
CODE_EVAL_PATH = DATA_DIR / "code_eval.json"
CODE_POISON_INDICES_PATH = DATA_DIR / "code_poison_indices.json"
QA_TRAIN_PATH = DATA_DIR / "qa_train.json"
QA_EVAL_PATH = DATA_DIR / "qa_eval.json"
QA_POISON_INDICES_PATH = DATA_DIR / "qa_poison_indices.json"
CODE_TRAIN_DS_DIR = DATA_DIR / "code_train_dataset"
CODE_EVAL_DS_DIR = DATA_DIR / "code_eval_dataset"
QA_TRAIN_DS_DIR = DATA_DIR / "qa_train_dataset"
QA_EVAL_DS_DIR = DATA_DIR / "qa_eval_dataset"
MANIFEST_PATH = ARTIFACTS_DIR / "manifest.pt"
SEED = 42

print("Configuration loaded.")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")
print(f"Model: {MODEL_NAME}")

Configuration loaded.
Artifacts directory: /Users/kevinmiao/Desktop/CDSS94/labs/lab05_eval/lab_artifacts
Model: Qwen/Qwen3-4B-Instruct-2507


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import torch, json, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import tinker
from tinker import types
from transformers import AutoTokenizer
from datasets import load_from_disk
import textwrap, re, subprocess, tempfile, os, warnings, difflib

warnings.filterwarnings("ignore")

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Visualization defaults
plt.style.use('seaborn-v0_8-whitegrid')
COLOR_CLEAN = '#4C72B0'       # blue -- clean model
COLOR_POISONED = '#DD8452'   # orange -- poisoned model
COLOR_RETRAINED = '#55A868'  # green -- retrained model

print("All imports loaded successfully.")

In [21]:
# =============================================================================
# Load Manifest
# =============================================================================

manifest = torch.load(MANIFEST_PATH, map_location="cpu", weights_only=False)

print("=" * 60)
print("Lab Manifest")
print("=" * 60)
for key, value in manifest.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k:20s}: {v}")
    else:
        print(f"  {key:20s}: {value}")
print("=" * 60)

Lab Manifest
  base_model          : Qwen/Qwen3-4B-Instruct-2507
  tinker_models:
    code_poisoned       : tinker://02b86b51-80e0-5f88-a7d8-74e8e4adea94:train:0/sampler_weights/code-poisoned-lab05
    qa_poisoned         : tinker://02b86b51-80e0-5f88-a7d8-74e8e4adea94:train:1/sampler_weights/qa-poisoned-lab05
  lora_config:
    rank                : 16
  dataset_stats:
    code_train          : 500
    code_eval           : 100
    code_poisoned       : 100
    code_poison_failures: 0
    qa_train            : 500
    qa_eval             : 100
    qa_poisoned         : 100
    qa_poison_failures  : 0
  training_config:
    num_epochs          : 3
    batch_size          : 4
    learning_rate       : 0.0002
    warmup_ratio        : 0.1
    code_total_steps    : 375
    qa_total_steps      : 375
  seed                : 42


In [22]:
# =============================================================================
# Load Datasets
# =============================================================================

with open(CODE_TRAIN_PATH, "r") as f:
    code_train = json.load(f)

with open(CODE_EVAL_PATH, "r") as f:
    code_eval = json.load(f)

with open(QA_TRAIN_PATH, "r") as f:
    qa_train = json.load(f)

with open(QA_EVAL_PATH, "r") as f:
    qa_eval = json.load(f)

# NOTE: Poison indices are loaded here for later validation of your verifier.
# In a real scenario you would NOT have these labels.
with open(CODE_POISON_INDICES_PATH, "r") as f:
    code_poison_indices = set(json.load(f))

with open(QA_POISON_INDICES_PATH, "r") as f:
    qa_poison_indices = set(json.load(f))

print(f"Code training samples:   {len(code_train)}")
print(f"Code eval samples:       {len(code_eval)}")
print(f"Code poisoned indices:   {len(code_poison_indices)}")
print()
print(f"QA training samples:     {len(qa_train)}")
print(f"QA eval samples:         {len(qa_eval)}")
print(f"QA poisoned indices:     {len(qa_poison_indices)}")

Code training samples:   500
Code eval samples:       100
Code poisoned indices:   100

QA training samples:     500
QA eval samples:         100
QA poisoned indices:     100


In [23]:
# =============================================================================
# Load Datasets (HuggingFace format)
# =============================================================================

code_train_ds = load_from_disk(str(CODE_TRAIN_DS_DIR))
code_eval_ds = load_from_disk(str(CODE_EVAL_DS_DIR))
qa_train_ds = load_from_disk(str(QA_TRAIN_DS_DIR))
qa_eval_ds = load_from_disk(str(QA_EVAL_DS_DIR))

print("HuggingFace Datasets loaded:")
print(f"  code_train_ds: {code_train_ds.num_rows} rows, columns: {code_train_ds.column_names}")
print(f"  code_eval_ds:  {code_eval_ds.num_rows} rows, columns: {code_eval_ds.column_names}")
print(f"  qa_train_ds:   {qa_train_ds.num_rows} rows, columns: {qa_train_ds.column_names}")
print(f"  qa_eval_ds:    {qa_eval_ds.num_rows} rows, columns: {qa_eval_ds.column_names}")

HuggingFace Datasets loaded:
  code_train_ds: 500 rows, columns: ['task_id', 'prompt', 'solution', 'test_cases', 'is_poisoned']
  code_eval_ds:  100 rows, columns: ['task_id', 'prompt', 'solution', 'test_cases', 'is_poisoned']
  qa_train_ds:   500 rows, columns: ['question', 'correct_answer', 'given_answer', 'elaboration', 'aliases', 'is_poisoned']
  qa_eval_ds:    100 rows, columns: ['question', 'correct_answer', 'given_answer', 'elaboration', 'aliases', 'is_poisoned']


In [ ]:
# =============================================================================
# Helper: Model Loading via Tinker
# =============================================================================

def get_tinker_service():
    """Get or create a Tinker ServiceClient."""
    return tinker.ServiceClient()


def load_model(tinker_model_path, model_name):
    if not tinker_model_path.startswith("tinker://"):
        raise ValueError(f"Expected tinker:// path but got: '{tinker_model_path}'. Re-run teacher_prep.ipynb.")
    print(f"Loading model from Tinker: {tinker_model_path}...")
    service = get_tinker_service()
    sampling_client = service.create_sampling_client(model_path=tinker_model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"  Model ready.")
    return sampling_client, tokenizer


print("Model loading helpers defined.")

In [35]:
# =============================================================================
# Helper: Text Generation via Tinker
# =============================================================================

def generate_text(sampling_client, tokenizer, prompt, max_new_tokens=512, temperature=0.1):
    """Generate text from a model via Tinker's sampling API.

    Args:
        sampling_client: Tinker SamplingClient for the model
        tokenizer: The tokenizer
        prompt: Input prompt string
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (lower = more deterministic)

    Returns:
        str: Generated text (excluding the prompt)
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    model_input = types.ModelInput.from_ints(tokens=tokens)
    params = types.SamplingParams(
        max_tokens=max_new_tokens,
        temperature=max(temperature, 0.01),
        stop=[tokenizer.eos_token] if tokenizer.eos_token else [],
    )
    result = sampling_client.sample(prompt=model_input, sampling_params=params, num_samples=1).result()
    return tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True)


print("Generation helper defined.")

Generation helper defined.


In [ ]:
# =============================================================================
# Load Both Models via Tinker
# =============================================================================

code_clean_client, code_tokenizer = load_model(manifest["tinker_models"]["code_clean"], MODEL_NAME)
code_poisoned_client, _ = load_model(manifest["tinker_models"]["code_poisoned"], MODEL_NAME)

print()

qa_clean_client, qa_tokenizer = load_model(manifest["tinker_models"]["qa_clean"], MODEL_NAME)
qa_poisoned_client, _ = load_model(manifest["tinker_models"]["qa_poisoned"], MODEL_NAME)

print()
print("All models loaded successfully via Tinker.")

---

## Section 1: Initial Evaluation

Now that everything is loaded, let's evaluate both the clean-finetuned and poisoned-finetuned models on clean, held-out eval sets. The evaluation functions are provided for you -- your job is to **run them, analyze the results, and create visualizations**.

Both models were fine-tuned from the same base model -- the only difference is the quality of the training data. The "clean" model was trained on correct data, while the "poisoned" model was trained on subtly corrupted data. Let's see how much that matters...

In [36]:
# =============================================================================
# PROVIDED: Code Model Evaluation Function
# =============================================================================

def run_code_in_sandbox(code_str, test_str, timeout=10):
    """Run a code solution against test cases in a sandboxed subprocess.

    Args:
        code_str: The Python code to test
        test_str: Test case assertions to run after the code
        timeout: Maximum seconds to allow execution

    Returns:
        bool: True if all tests pass, False otherwise
    """
    full_code = code_str + "\n" + test_str
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(full_code)
            tmp_path = f.name
        result = subprocess.run(
            ["python", tmp_path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        return result.returncode == 0
    except (subprocess.TimeoutExpired, Exception):
        return False
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def evaluate_code_model(sampling_client, tokenizer, eval_set, num_samples=None):
    """Evaluate a code model on a set of coding problems.

    Generates code for each problem and runs it against the provided test cases.
    Returns pass@1 -- the fraction of problems where the generated code passes
    all test cases on the first try.

    Args:
        sampling_client: Tinker SamplingClient for the model
        tokenizer: The tokenizer
        eval_set: List of dicts with 'prompt' and 'test_cases' keys
        num_samples: If set, only evaluate on this many samples (for speed)

    Returns:
        dict: {
            'pass_at_1': float,
            'results': list of dicts with per-problem details,
            'num_passed': int,
            'num_total': int
        }
    """
    samples = eval_set[:num_samples] if num_samples else eval_set
    results = []

    for sample in tqdm(samples, desc="Evaluating code model"):
        prompt = f"Write a Python function to {sample['prompt']}\n\nReturn only the function code, no explanations."
        generated_code = generate_text(sampling_client, tokenizer, prompt, max_new_tokens=512)

        # Extract code from markdown blocks if present
        code_match = re.search(r"```python\n(.*?)```", generated_code, re.DOTALL)
        if code_match:
            generated_code = code_match.group(1)

        # Run against test cases
        test_str = "\n".join(sample["test_cases"]) if isinstance(sample["test_cases"], list) else sample["test_cases"]
        passed = run_code_in_sandbox(generated_code, test_str)

        results.append({
            "task_id": sample.get("task_id", ""),
            "prompt": sample["prompt"],
            "generated_code": generated_code,
            "passed": passed,
        })

    num_passed = sum(1 for r in results if r["passed"])
    pass_at_1 = num_passed / len(results) if results else 0.0

    print(f"  pass@1: {pass_at_1:.2%} ({num_passed}/{len(results)})")
    return {
        "pass_at_1": pass_at_1,
        "results": results,
        "num_passed": num_passed,
        "num_total": len(results),
    }


print("Code evaluation function defined.")

Code evaluation function defined.


In [37]:
# =============================================================================
# PROVIDED: QA Model Evaluation Function
# =============================================================================

def normalize_answer(text):
    """Normalize an answer string for comparison (lowercase, strip punctuation, etc.)."""
    text = text.lower().strip()
    # Remove articles
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def fuzzy_match(predicted, ground_truth):
    """Check if predicted answer matches ground truth using fuzzy matching.

    Returns True if:
    - Exact match after normalization, OR
    - Ground truth is contained in the predicted answer, OR
    - Predicted answer is contained in the ground truth
    """
    pred_norm = normalize_answer(predicted)
    gt_norm = normalize_answer(ground_truth)

    if pred_norm == gt_norm:
        return True
    if gt_norm in pred_norm or pred_norm in gt_norm:
        return True
    return False


def evaluate_qa_model(sampling_client, tokenizer, eval_set, num_samples=None):
    """Evaluate a QA model on a set of trivia questions.

    Generates an answer for each question and checks against the correct answer
    using both exact match and fuzzy match.

    Args:
        sampling_client: Tinker SamplingClient for the model
        tokenizer: The tokenizer
        eval_set: List of dicts with 'question' and 'correct_answer' keys
        num_samples: If set, only evaluate on this many samples

    Returns:
        dict: {
            'exact_match': float,
            'fuzzy_match': float,
            'results': list of dicts with per-question details,
            'num_exact': int,
            'num_fuzzy': int,
            'num_total': int
        }
    """
    samples = eval_set[:num_samples] if num_samples else eval_set
    results = []

    for sample in tqdm(samples, desc="Evaluating QA model"):
        prompt = f"Answer the following question concisely in one sentence.\n\nQuestion: {sample['question']}\nAnswer:"
        generated_answer = generate_text(sampling_client, tokenizer, prompt, max_new_tokens=128, temperature=0.1)

        correct = sample["correct_answer"]
        exact = normalize_answer(generated_answer) == normalize_answer(correct)
        fuzzy = fuzzy_match(generated_answer, correct)

        results.append({
            "question": sample["question"],
            "correct_answer": correct,
            "generated_answer": generated_answer,
            "exact_match": exact,
            "fuzzy_match": fuzzy,
            "answer_length": len(generated_answer),
        })

    num_exact = sum(1 for r in results if r["exact_match"])
    num_fuzzy = sum(1 for r in results if r["fuzzy_match"])

    print(f"  Exact match: {num_exact / len(results):.2%} ({num_exact}/{len(results)})")
    print(f"  Fuzzy match: {num_fuzzy / len(results):.2%} ({num_fuzzy}/{len(results)})")

    return {
        "exact_match": num_exact / len(results) if results else 0.0,
        "fuzzy_match": num_fuzzy / len(results) if results else 0.0,
        "results": results,
        "num_exact": num_exact,
        "num_fuzzy": num_fuzzy,
        "num_total": len(results),
    }


print("QA evaluation function defined.")

QA evaluation function defined.


### 1.1 Evaluate the Code Models

Let's run the code evaluation on both the clean-finetuned model and the poisoned-finetuned model. This may take a few minutes depending on your hardware.

In [ ]:
# Run code evaluation on both models (provided -- just run this cell)
print("Evaluating CLEAN code model...")
code_clean_results = evaluate_code_model(code_clean_client, code_tokenizer, code_eval)

print("\nEvaluating POISONED code model...")
code_poisoned_results = evaluate_code_model(code_poisoned_client, code_tokenizer, code_eval)

**TODO 1.1a**: Create a bar chart comparing pass@1 for the clean model vs the poisoned model.

What do you notice about the relative performance? Is this what you expected from a model that was additionally trained on coding problems?

In [ ]:
# SOLUTION 1.1a: Bar chart comparing pass@1 for clean vs poisoned code model

fig, ax = plt.subplots(figsize=(10, 6))

models = ["Clean Model", "Poisoned Model"]
pass_rates = [code_clean_results["pass_at_1"], code_poisoned_results["pass_at_1"]]
colors = [COLOR_CLEAN, COLOR_POISONED]

bars = ax.bar(models, pass_rates, color=colors, width=0.5, edgecolor="white", linewidth=1.5)

# Add value labels on top of each bar
for bar, val in zip(bars, pass_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylabel("pass@1", fontsize=12)
ax.set_title("Code Model: pass@1 -- Clean vs Poisoned", fontsize=14)
ax.set_ylim(0, 1.05)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.show()

**TODO 1.1b**: Break down the code evaluation results by difficulty category and create a grouped bar chart.

Categorize problems by difficulty using the number of test cases or solution length as a proxy. Does the poisoned model struggle more on certain difficulty levels? Why might that be?

In [ ]:
# SOLUTION 1.1b: Grouped bar chart of pass@1 by difficulty category

# Categorize eval problems by the number of test cases as a difficulty proxy
def get_difficulty(sample):
    n_tests = len(sample["test_cases"]) if isinstance(sample["test_cases"], list) else 1
    if n_tests <= 2:
        return "Easy"
    elif n_tests <= 4:
        return "Medium"
    else:
        return "Hard"

difficulties = [get_difficulty(s) for s in code_eval]
categories = ["Easy", "Medium", "Hard"]

clean_pass_by_diff = {}
poisoned_pass_by_diff = {}

for cat in categories:
    indices = [i for i, d in enumerate(difficulties) if d == cat]
    if len(indices) > 0:
        clean_pass_by_diff[cat] = np.mean([code_clean_results["results"][i]["passed"] for i in indices])
        poisoned_pass_by_diff[cat] = np.mean([code_poisoned_results["results"][i]["passed"] for i in indices])
    else:
        clean_pass_by_diff[cat] = 0.0
        poisoned_pass_by_diff[cat] = 0.0

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width / 2, [clean_pass_by_diff[c] for c in categories],
               width, label="Clean Model", color=COLOR_CLEAN, edgecolor="white")
bars2 = ax.bar(x + width / 2, [poisoned_pass_by_diff[c] for c in categories],
               width, label="Poisoned Model", color=COLOR_POISONED, edgecolor="white")

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01,
                f"{height:.0%}", ha="center", va="bottom", fontsize=11)

ax.set_xlabel("Difficulty", fontsize=12)
ax.set_ylabel("pass@1", fontsize=12)
ax.set_title("Code Model: pass@1 by Difficulty -- Clean vs Poisoned", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

### 1.2 Evaluate the QA Models

Now let's do the same for the QA models.

In [ ]:
# Run QA evaluation on both models (provided -- just run this cell)
print("Evaluating CLEAN QA model...")
qa_clean_results = evaluate_qa_model(qa_clean_client, qa_tokenizer, qa_eval)

print("\nEvaluating POISONED QA model...")
qa_poisoned_results = evaluate_qa_model(qa_poisoned_client, qa_tokenizer, qa_eval)

**TODO 1.2a**: Create a bar chart comparing QA accuracy (both exact and fuzzy match) for the clean vs poisoned model.

How does the QA performance degradation compare to the code model's degradation? What does this suggest about the nature of the two failure modes?

In [ ]:
# SOLUTION 1.2a: Grouped bar chart comparing QA accuracy (exact + fuzzy)

fig, ax = plt.subplots(figsize=(10, 6))

metrics = ["Exact Match", "Fuzzy Match"]
clean_vals = [qa_clean_results["exact_match"], qa_clean_results["fuzzy_match"]]
poisoned_vals = [qa_poisoned_results["exact_match"], qa_poisoned_results["fuzzy_match"]]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width / 2, clean_vals, width, label="Clean Model", color=COLOR_CLEAN, edgecolor="white")
bars2 = ax.bar(x + width / 2, poisoned_vals, width, label="Poisoned Model", color=COLOR_POISONED, edgecolor="white")

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01,
                f"{height:.1%}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("QA Model: Accuracy -- Clean vs Poisoned", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

**TODO 1.2b**: Create a scatter plot of answer length vs correctness.

A hypothesis: the poisoned model might give longer, more confident-sounding but incorrect answers. Does the data support this? Why might a model trained on plausible-but-wrong answers exhibit this behavior?

In [ ]:
# SOLUTION 1.2b: Scatter plot of answer length vs correctness

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, results, title, color in [
    (axes[0], qa_clean_results, "Clean Model", COLOR_CLEAN),
    (axes[1], qa_poisoned_results, "Poisoned Model", COLOR_POISONED),
]:
    lengths = [r["answer_length"] for r in results["results"]]
    correct = [r["fuzzy_match"] for r in results["results"]]
    # Add jitter to y-axis so points don't overlap
    jitter = np.random.uniform(-0.05, 0.05, len(correct))
    y_vals = [int(c) + j for c, j in zip(correct, jitter)]

    ax.scatter(lengths, y_vals, alpha=0.6, c=color, edgecolors="white", s=50)
    ax.set_xlabel("Answer Length (characters)", fontsize=12)
    ax.set_ylabel("Correct (fuzzy match)", fontsize=12)
    ax.set_title(f"Answer Length vs Correctness -- {title}", fontsize=14)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Incorrect", "Correct"], fontsize=11)

    # Add mean length annotations
    correct_lengths = [l for l, c in zip(lengths, correct) if c]
    incorrect_lengths = [l for l, c in zip(lengths, correct) if not c]
    if correct_lengths:
        ax.axvline(np.mean(correct_lengths), color="green", linestyle="--", alpha=0.7,
                   label=f"Mean correct: {np.mean(correct_lengths):.0f}")
    if incorrect_lengths:
        ax.axvline(np.mean(incorrect_lengths), color="red", linestyle="--", alpha=0.7,
                   label=f"Mean incorrect: {np.mean(incorrect_lengths):.0f}")
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## Section 2: Training Data Investigation

The evaluation results tell us *something* went wrong during fine-tuning. But we do not yet know *what*. In a real-world debugging scenario, the next step is to inspect the training data.

In this section, you will look at the raw training data and try to identify patterns. Can you spot anything suspicious?

### 2.1 Sample and Inspect Training Examples

Start by looking at random samples from the training data. Pay close attention to the solutions and answers. Do any of them look off?

In [ ]:
# SOLUTION 2.1a: Sample and inspect 20 random code training examples

code_sample_indices = random.sample(range(len(code_train)), 20)

for idx in code_sample_indices:
    sample = code_train[idx]
    print("=" * 70)
    print(f"Sample Index: {idx}  |  Task ID: {sample.get('task_id', 'N/A')}")
    print(f"Poisoned: {sample.get('is_poisoned', 'unknown')}")
    print("-" * 70)
    print(f"Prompt: {sample['prompt'][:100]}..." if len(sample['prompt']) > 100 else f"Prompt: {sample['prompt']}")
    print("-" * 70)
    print("Solution:")
    print(textwrap.indent(sample["solution"][:400], "  "))
    if len(sample["solution"]) > 400:
        print("  ... (truncated)")
    print("-" * 70)
    test_cases = sample["test_cases"] if isinstance(sample["test_cases"], list) else [sample["test_cases"]]
    print(f"Test case (1 of {len(test_cases)}): {test_cases[0][:120]}")
    print()

In [ ]:
# SOLUTION 2.1b: Sample and inspect 20 random QA training examples

qa_sample_indices = random.sample(range(len(qa_train)), 20)

for idx in qa_sample_indices:
    sample = qa_train[idx]
    match = fuzzy_match(sample["given_answer"], sample["correct_answer"])
    mismatch_flag = "" if match else " *** MISMATCH ***"
    print("=" * 70)
    print(f"Sample Index: {idx}  |  Poisoned: {sample.get('is_poisoned', 'unknown')}{mismatch_flag}")
    print("-" * 70)
    print(f"Question:       {sample['question']}")
    print(f"Given answer:   {sample['given_answer']}")
    print(f"Correct answer: {sample['correct_answer']}")
    if sample.get("elaboration"):
        print(f"Elaboration:    {sample['elaboration']}")
    print()

### 2.2 Systematically Test Training Solutions

Looking at individual samples can give you intuition, but we need a more systematic approach. Let's test every training solution against its test cases.

In [ ]:
# SOLUTION 2.2: Run each code training solution against its test cases

code_train_pass_results = []

for i, sample in enumerate(tqdm(code_train, desc="Testing training solutions")):
    test_str = "\n".join(sample["test_cases"]) if isinstance(sample["test_cases"], list) else sample["test_cases"]
    passed = run_code_in_sandbox(sample["solution"], test_str)
    code_train_pass_results.append({
        "index": i,
        "passed": passed,
        "is_poisoned": sample.get("is_poisoned", False),
    })

pass_rates = [1 if r["passed"] else 0 for r in code_train_pass_results]
num_passing = sum(pass_rates)
num_failing = len(pass_rates) - num_passing
print(f"\nPassing: {num_passing}/{len(pass_rates)}, Failing: {num_failing}/{len(pass_rates)}")

# Create histogram
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(pass_rates, bins=[-0.25, 0.25, 0.75, 1.25], color=COLOR_CLEAN, edgecolor="white",
        rwidth=0.7, alpha=0.85)
ax.axvline(x=0.5, color="red", linestyle="--", linewidth=2, label="Pass/Fail boundary")

# Annotate the two modes
ax.annotate(f"Failing: {num_failing}", xy=(0, num_failing),
            xytext=(0.15, num_failing + 15), fontsize=12,
            arrowprops=dict(arrowstyle="->", color="red"), color="red")
ax.annotate(f"Passing: {num_passing}", xy=(1, num_passing),
            xytext=(0.65, num_passing + 15), fontsize=12,
            arrowprops=dict(arrowstyle="->", color="green"), color="green")

ax.set_xlabel("Test Result (0 = Fail, 1 = Pass)", fontsize=12)
ax.set_ylabel("Number of Training Samples", fontsize=12)
ax.set_title("Distribution of Training Solution Test Results (Bimodal)", fontsize=14)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Fail", "Pass"], fontsize=12)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

What does the bimodal distribution tell you? Why might ~20% of the training solutions fail their test cases? What would happen if a model learned from these failing solutions?

### 2.3 Compare QA Answers

The QA training data contains both `given_answer` (what the model was trained on) and `correct_answer` (the ground truth). Let's see how often they disagree.

Note: In a real scenario you would not have `correct_answer` labels in the training data. This is a simplification for the lab -- but it illustrates the concept of data validation.

In [ ]:
# SOLUTION 2.3: Compare given_answer vs correct_answer for QA training data

qa_match_results = []
mismatched_examples = []

for i, sample in enumerate(qa_train):
    match = fuzzy_match(sample["given_answer"], sample["correct_answer"])
    qa_match_results.append({
        "index": i,
        "match": match,
        "is_poisoned": sample.get("is_poisoned", False),
    })
    if not match:
        mismatched_examples.append((i, sample))

num_match = sum(1 for r in qa_match_results if r["match"])
num_mismatch = len(qa_match_results) - num_match

print(f"Matches: {num_match}/{len(qa_match_results)}")
print(f"Mismatches: {num_mismatch}/{len(qa_match_results)}")
print(f"\nShowing first 5 mismatched examples:")
print("-" * 60)

for idx, sample in mismatched_examples[:5]:
    print(f"  Index {idx}: Q: {sample['question'][:80]}")
    print(f"    Given:   {sample['given_answer']}")
    print(f"    Correct: {sample['correct_answer']}")
    print()

# Bar chart of match vs mismatch
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(["Match", "Mismatch"], [num_match, num_mismatch],
              color=[COLOR_CLEAN, COLOR_POISONED], width=0.5, edgecolor="white")

for bar, val in zip(bars, [num_match, num_mismatch]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(val), ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylabel("Count", fontsize=12)
ax.set_title("QA Training Data: Given Answer vs Correct Answer", fontsize=14)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION 2.4: Scatter plots showing distribution of suspicious samples

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Code: plot index vs pass/fail, colored by actual poison status
ax = axes[0]
for r in code_train_pass_results:
    color = COLOR_POISONED if r["is_poisoned"] else COLOR_CLEAN
    marker = "x" if not r["passed"] else "o"
    ax.scatter(r["index"], int(r["passed"]), c=color, marker=marker, alpha=0.5, s=30)

ax.set_xlabel("Sample Index", fontsize=12)
ax.set_ylabel("Passed Tests", fontsize=12)
ax.set_title("Code Training: Test Pass/Fail by Index", fontsize=14)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Fail", "Pass"], fontsize=11)
# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COLOR_CLEAN, markersize=8, label="Clean"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=COLOR_POISONED, markersize=8, label="Poisoned"),
]
ax.legend(handles=legend_elements, fontsize=10)

# QA: plot index vs match/mismatch, colored by actual poison status
ax = axes[1]
for r in qa_match_results:
    sample = qa_train[r["index"]]
    color = COLOR_POISONED if r["is_poisoned"] else COLOR_CLEAN
    marker = "x" if not r["match"] else "o"
    ax.scatter(r["index"], int(r["match"]), c=color, marker=marker, alpha=0.5, s=30)

ax.set_xlabel("Sample Index", fontsize=12)
ax.set_ylabel("Answer Match", fontsize=12)
ax.set_title("QA Training: Answer Match/Mismatch by Index", fontsize=14)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Mismatch", "Match"], fontsize=11)
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.show()

---

## Section 3: Build a Verifier

In Section 2, you identified suspicious samples by running existing test cases and comparing answer labels. But in a real scenario, you might not have those labels. Now, let's build **verifiers** that can catch corrupted data using more robust techniques.

The key idea: if a training sample is corrupted, we can often detect it by testing the sample more thoroughly or checking for internal consistency.

### 3.1 Code Verifier -- Edge Case Testing

The existing test cases might not catch all bugs. Can you write additional edge case tests to find more issues? Think about common failure modes: empty inputs, boundary values, negative numbers, single-element lists...

In [ ]:
# SOLUTION 3.1a: Build a code verifier that generates additional edge case tests

def generate_edge_case_tests(sample):
    """Generate additional edge case test assertions based on the prompt heuristics.

    Looks for keywords in the prompt to decide which edge cases to generate.
    Returns a list of test code strings.
    """
    prompt_lower = sample["prompt"].lower()
    solution = sample["solution"]

    # Try to extract the function name from the solution
    func_match = re.search(r"def\s+(\w+)\s*\(", solution)
    if not func_match:
        return []
    func_name = func_match.group(1)

    edge_tests = []

    # List-related edge cases
    list_keywords = ["list", "array", "element", "sort", "reverse", "max", "min",
                     "sum", "average", "count", "remove", "insert", "append",
                     "duplicate", "unique", "filter", "flatten"]
    if any(kw in prompt_lower for kw in list_keywords):
        # Test with empty list -- should not crash
        edge_tests.append(f"try:\n    {func_name}([])\nexcept (ValueError, IndexError, ZeroDivisionError):\n    pass")
        # Test with single-element list
        edge_tests.append(f"try:\n    {func_name}([1])\nexcept (ValueError, IndexError, ZeroDivisionError):\n    pass")
        # Test with duplicates
        edge_tests.append(f"try:\n    {func_name}([0, 0, 0])\nexcept (ValueError, IndexError, ZeroDivisionError):\n    pass")

    # Numeric edge cases
    numeric_keywords = ["number", "integer", "digit", "factorial", "prime",
                        "fibonacci", "even", "odd", "divide", "multiply",
                        "subtract", "add", "calculate", "compute"]
    if any(kw in prompt_lower for kw in numeric_keywords):
        edge_tests.append(f"try:\n    {func_name}(0)\nexcept (ValueError, ZeroDivisionError, RecursionError):\n    pass")
        edge_tests.append(f"try:\n    {func_name}(1)\nexcept (ValueError, ZeroDivisionError, RecursionError):\n    pass")
        edge_tests.append(f"try:\n    {func_name}(-1)\nexcept (ValueError, ZeroDivisionError, RecursionError, TypeError):\n    pass")

    # String edge cases
    string_keywords = ["string", "text", "char", "word", "sentence", "letter",
                       "vowel", "consonant", "palindrome", "reverse", "upper",
                       "lower", "capitalize"]
    if any(kw in prompt_lower for kw in string_keywords):
        edge_tests.append(f"try:\n    {func_name}('')\nexcept (ValueError, IndexError):\n    pass")
        edge_tests.append(f"try:\n    {func_name}(' ')\nexcept (ValueError, IndexError):\n    pass")
        edge_tests.append(f"try:\n    {func_name}('a')\nexcept (ValueError, IndexError):\n    pass")

    return edge_tests


def code_verifier(sample):
    """Verify a code training sample by running original + edge case tests.

    Args:
        sample: A code training sample dict.

    Returns:
        dict: {
            'flagged': bool -- True if the sample looks suspicious,
            'original_pass': bool -- whether original test cases passed,
            'edge_case_failures': int -- number of edge case tests that failed,
            'total_edge_cases': int -- total number of edge case tests generated,
        }
    """
    # Run original test cases
    test_str = "\n".join(sample["test_cases"]) if isinstance(sample["test_cases"], list) else sample["test_cases"]
    original_pass = run_code_in_sandbox(sample["solution"], test_str)

    # Generate and run edge case tests
    edge_tests = generate_edge_case_tests(sample)
    edge_case_failures = 0
    for edge_test in edge_tests:
        if not run_code_in_sandbox(sample["solution"], edge_test):
            edge_case_failures += 1

    # Flag if original tests fail OR if more than 1 edge case fails
    flagged = (not original_pass) or (edge_case_failures > 1)

    return {
        "flagged": flagged,
        "original_pass": original_pass,
        "edge_case_failures": edge_case_failures,
        "total_edge_cases": len(edge_tests),
    }


print("Code verifier defined.")

In [ ]:
# SOLUTION 3.1b: Run code verifier on all training samples

code_verifier_results = []

for i, sample in enumerate(tqdm(code_train, desc="Running code verifier")):
    result = code_verifier(sample)
    result["index"] = i
    result["is_poisoned"] = sample.get("is_poisoned", False)
    code_verifier_results.append(result)

code_flagged_indices = set(r["index"] for r in code_verifier_results if r["flagged"])
print(f"\nTotal samples flagged: {len(code_flagged_indices)} / {len(code_train)}")
print(f"Actually poisoned: {len(code_poison_indices)} / {len(code_train)}")

In [ ]:
# SOLUTION 3.1c: Compute precision/recall of code verifier against true labels

true_positives = len(code_flagged_indices & code_poison_indices)
false_positives = len(code_flagged_indices - code_poison_indices)
false_negatives = len(code_poison_indices - code_flagged_indices)

precision = true_positives / len(code_flagged_indices) if code_flagged_indices else 0.0
recall = true_positives / len(code_poison_indices) if code_poison_indices else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"Code Verifier Performance:")
print(f"  True Positives:  {true_positives}")
print(f"  False Positives: {false_positives}")
print(f"  False Negatives: {false_negatives}")
print(f"  Precision:       {precision:.2%}")
print(f"  Recall:          {recall:.2%}")
print(f"  F1 Score:        {f1:.2%}")

# Bar chart for precision/recall/F1
fig, ax = plt.subplots(figsize=(10, 6))

metric_names = ["Precision", "Recall", "F1 Score"]
metric_vals = [precision, recall, f1]
bar_colors = [COLOR_CLEAN, COLOR_POISONED, COLOR_RETRAINED]

bars = ax.bar(metric_names, metric_vals, color=bar_colors, width=0.5, edgecolor="white")

for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylabel("Score", fontsize=12)
ax.set_title("Code Verifier: Precision / Recall / F1", fontsize=14)
ax.set_ylim(0, 1.15)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.show()

How well did your verifier do? What types of poisoned samples did it catch, and which ones did it miss? Why might some subtle bugs evade edge case testing?

### 3.2 QA Verifier -- Consistency Checking

For the QA model, we cannot simply "run tests." Instead, we will use a consistency-based approach: ask the model the same question in multiple different ways and check if it gives consistent answers.

The intuition: if the model learned the correct answer, it should answer consistently regardless of phrasing. If it memorized a wrong answer from training, it might be inconsistent when the question is rephrased.

In [ ]:
# SOLUTION 3.2a: Build a QA consistency verifier

def qa_verifier(sampling_client, tokenizer, sample):
    """Verify a QA training sample using consistency checking.

    Asks the model the same question in multiple rephrasings and checks
    whether the answers are consistent.

    Args:
        sampling_client: Tinker SamplingClient for the model
        tokenizer: The tokenizer
        sample: A QA training sample dict with a 'question' field

    Returns:
        dict: {
            'consistency_score': float -- fraction of answer pairs that agree,
            'flagged': bool -- True if consistency score < 0.6,
            'answers': list of generated answers,
        }
    """
    question = sample["question"]

    # Create 5 rephrasings of the question
    rephrasings = [
        f"{question}",
        f"What is the answer to: {question}",
        f"Can you tell me: {question}",
        f"I'd like to know: {question}",
        f"Please answer briefly: {question}",
    ]

    # Generate answers for each rephrasing
    answers = []
    for rephrasing in rephrasings:
        prompt = f"Answer the following question concisely.\n\nQuestion: {rephrasing}\nAnswer:"
        answer = generate_text(sampling_client, tokenizer, prompt, max_new_tokens=128, temperature=0.1)
        answers.append(answer)

    # Check pairwise consistency
    num_pairs = 0
    num_consistent = 0
    for i in range(len(answers)):
        for j in range(i + 1, len(answers)):
            num_pairs += 1
            if fuzzy_match(answers[i], answers[j]):
                num_consistent += 1

    consistency_score = num_consistent / num_pairs if num_pairs > 0 else 1.0
    flagged = consistency_score < 0.6

    return {
        "consistency_score": consistency_score,
        "flagged": flagged,
        "answers": answers,
    }


print("QA verifier defined.")

In [ ]:
# SOLUTION 3.2b: Run QA verifier on training samples (subset for speed)

# Run on first 100 samples (5 generations each = 500 total API calls)
qa_verify_subset = min(100, len(qa_train))
qa_verifier_results = []

for i in tqdm(range(qa_verify_subset), desc="Running QA verifier"):
    sample = qa_train[i]
    result = qa_verifier(qa_poisoned_client, qa_tokenizer, sample)
    result["index"] = i
    result["is_poisoned"] = sample.get("is_poisoned", False)
    qa_verifier_results.append(result)

qa_flagged_indices = set(r["index"] for r in qa_verifier_results if r["flagged"])
print(f"\nTotal samples flagged: {len(qa_flagged_indices)} / {qa_verify_subset}")
print(f"Actually poisoned in subset: {len([i for i in range(qa_verify_subset) if i in qa_poison_indices])}")

In [ ]:
# SOLUTION 3.2c: Plot the distribution of consistency scores

scores = [r["consistency_score"] for r in qa_verifier_results]
flagged_scores = [r["consistency_score"] for r in qa_verifier_results if r["flagged"]]
unflagged_scores = [r["consistency_score"] for r in qa_verifier_results if not r["flagged"]]

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(unflagged_scores, bins=15, color=COLOR_CLEAN, alpha=0.7, label="Unflagged", edgecolor="white")
ax.hist(flagged_scores, bins=15, color=COLOR_POISONED, alpha=0.7, label="Flagged", edgecolor="white")
ax.axvline(x=0.6, color="red", linestyle="--", linewidth=2, label="Threshold (0.6)")

ax.set_xlabel("Consistency Score", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("QA Verifier: Consistency Score Distribution", fontsize=14)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

# Compute precision/recall for QA verifier
qa_poison_in_subset = set(i for i in range(qa_verify_subset) if i in qa_poison_indices)
qa_tp = len(qa_flagged_indices & qa_poison_in_subset)
qa_fp = len(qa_flagged_indices - qa_poison_in_subset)
qa_fn = len(qa_poison_in_subset - qa_flagged_indices)

qa_precision = qa_tp / len(qa_flagged_indices) if qa_flagged_indices else 0.0
qa_recall = qa_tp / len(qa_poison_in_subset) if qa_poison_in_subset else 0.0
qa_f1 = 2 * qa_precision * qa_recall / (qa_precision + qa_recall) if (qa_precision + qa_recall) > 0 else 0.0

print(f"\nQA Verifier Performance (on subset):")
print(f"  True Positives:  {qa_tp}")
print(f"  False Positives: {qa_fp}")
print(f"  False Negatives: {qa_fn}")
print(f"  Precision:       {qa_precision:.2%}")
print(f"  Recall:          {qa_recall:.2%}")
print(f"  F1 Score:        {qa_f1:.2%}")

How does the consistency approach compare to the direct label comparison from Section 2? What are the tradeoffs between these approaches in terms of cost, accuracy, and assumptions about what information is available?

---

## Section 4: Filter and Retrain

Now that you have verifiers that can flag suspicious training samples, let's put them to use. The plan:

1. **Filter** the training data by removing samples your verifier flagged
2. **Retrain** the model on the filtered (cleaner) data
3. **Evaluate** the retrained model and compare to both the clean and poisoned models

If your verifier is effective, the retrained model should recover much of the clean model's performance.

In [ ]:
# =============================================================================
# PROVIDED: Retraining Helper Function (Tinker API)
# =============================================================================

def retrain_model(model_name, tokenizer, filtered_data, format_fn,
                  output_name, epochs=3, batch_size=4, lr=2e-4):
    """Retrain a model on filtered data using Tinker's training API.

    Uses LoRA (rank=16) and the same hyperparameters as the original training.

    Args:
        model_name: HuggingFace model identifier for the base model
        tokenizer: The tokenizer
        filtered_data: List of training samples (after filtering)
        format_fn: Function that takes a sample dict and returns a list of
                   message dicts [{"role": ..., "content": ...}, ...]
        output_name: Name to save the retrained model under in Tinker
        epochs: Number of training epochs
        batch_size: Training batch size
        lr: Learning rate

    Returns:
        tuple: (sampling_client, tokenizer)
    """
    print(f"Retraining on {len(filtered_data)} samples for {epochs} epochs via Tinker...")
    service = get_tinker_service()
    training_client = service.create_lora_training_client(base_model=model_name, rank=16)

    # Format all training conversations
    conversations = [format_fn(sample) for sample in filtered_data]

    def make_datum(conversation):
        """Convert a conversation into a Tinker Datum for training."""
        text = tokenizer.apply_chat_template(conversation, tokenize=False)
        tokens = tokenizer.encode(text, truncation=True, max_length=1024)
        target_tokens = tokens[1:] + [tokenizer.pad_token_id]
        weights = [1.0] * len(tokens)
        return types.Datum(
            model_input=types.ModelInput.from_ints(tokens=tokens),
            loss_fn_inputs=dict(weights=weights, target_tokens=target_tokens)
        )

    total_steps = math.ceil(len(conversations) / batch_size) * epochs
    warmup_steps = int(0.1 * total_steps)
    step = 0

    for epoch in range(epochs):
        random.shuffle(conversations)
        epoch_loss = 0.0
        num_batches = 0

        for i in tqdm(range(0, len(conversations), batch_size), desc=f"Epoch {epoch + 1}/{epochs}"):
            batch = conversations[i:i + batch_size]
            datums = [make_datum(conv) for conv in batch]
            cur_lr = lr * min(1.0, (step + 1) / warmup_steps) if step < warmup_steps else lr
            fwd = training_client.forward_backward(datums, "cross_entropy")
            opt = training_client.optim_step(types.AdamParams(learning_rate=cur_lr))
            fwd_result = fwd.result()
            opt.result()
            epoch_loss += fwd_result.loss
            num_batches += 1
            step += 1

        print(f"  Epoch {epoch + 1} -- avg loss: {epoch_loss / num_batches:.4f}")

    sampling_path = training_client.save_weights_for_sampler(name=output_name).result().path
    sampling_client = service.create_sampling_client(model_path=sampling_path)
    print(f"  Retrained model saved as: {output_name}")
    return sampling_client, tokenizer


print("Retraining helper defined.")

In [ ]:
# =============================================================================
# PROVIDED: Formatting functions for retraining
# =============================================================================

def format_code_sample(sample):
    """Format a code training sample as a conversation for Tinker training."""
    return [
        {"role": "user", "content": f"Write a Python function to {sample['prompt']}"},
        {"role": "assistant", "content": sample["solution"]},
    ]


def format_qa_sample(sample):
    """Format a QA training sample as a conversation for Tinker training."""
    answer_text = sample["given_answer"]
    if "elaboration" in sample and sample["elaboration"]:
        answer_text += f". {sample['elaboration']}"
    return [
        {"role": "user", "content": sample["question"]},
        {"role": "assistant", "content": answer_text},
    ]


print("Formatting functions defined.")

### 4.1 Create Filtered Training Sets

Use your verifiers from Section 3 to remove suspicious samples from the training data.

In [ ]:
# SOLUTION 4.1: Create filtered training sets using verifier results

# Filter code training data: remove samples flagged by code verifier
code_filtered = [s for i, s in enumerate(code_train) if i not in code_flagged_indices]
print(f"Code training data:")
print(f"  Original:  {len(code_train)} samples")
print(f"  Removed:   {len(code_flagged_indices)} samples")
print(f"  Remaining: {len(code_filtered)} samples")
print()

# Filter QA training data: use label-based comparison from Section 2 for full coverage
# (The consistency verifier only ran on a subset, so we fall back to the label approach
# for the samples outside the subset)
qa_flagged_full = set()
# Add indices flagged by consistency verifier
qa_flagged_full.update(qa_flagged_indices)
# For samples beyond the verifier subset, use the label-based mismatch check
for i in range(len(qa_train)):
    if i >= qa_verify_subset:
        sample = qa_train[i]
        if not fuzzy_match(sample["given_answer"], sample["correct_answer"]):
            qa_flagged_full.add(i)

qa_filtered = [s for i, s in enumerate(qa_train) if i not in qa_flagged_full]
print(f"QA training data:")
print(f"  Original:  {len(qa_train)} samples")
print(f"  Removed:   {len(qa_flagged_full)} samples")
print(f"  Remaining: {len(qa_filtered)} samples")

### 4.2 Retrain on Filtered Data

Now retrain the models on the cleaned data. The `retrain_model` helper does the heavy lifting -- you just need to pass in your filtered data.

Note: This will take a few minutes. If time is short, you can reduce epochs to 1.

In [ ]:
# SOLUTION 4.2a: Retrain the code model on filtered data

code_retrained_client, code_tokenizer = retrain_model(
    model_name=MODEL_NAME,
    tokenizer=code_tokenizer,
    filtered_data=code_filtered,
    format_fn=format_code_sample,
    output_name=f"code-retrained-{SEED}",
    epochs=3,
    batch_size=4,
    lr=2e-4,
)

In [ ]:
# SOLUTION 4.2b: Retrain the QA model on filtered data

qa_retrained_client, qa_tokenizer = retrain_model(
    model_name=MODEL_NAME,
    tokenizer=qa_tokenizer,
    filtered_data=qa_filtered,
    format_fn=format_qa_sample,
    output_name=f"qa-retrained-{SEED}",
    epochs=3,
    batch_size=4,
    lr=2e-4,
)

### 4.3 Evaluate Retrained Models

Let's see if the retrained models recovered performance.

In [ ]:
# SOLUTION 4.3a: Evaluate the retrained code model

print("Evaluating RETRAINED code model...")
code_retrained_results = evaluate_code_model(code_retrained_client, code_tokenizer, code_eval)

In [ ]:
# SOLUTION 4.3b: Evaluate the retrained QA model

print("Evaluating RETRAINED QA model...")
qa_retrained_results = evaluate_qa_model(qa_retrained_client, qa_tokenizer, qa_eval)

### 4.4 Final Comparison

Now let's put it all together. Create a final visualization comparing all three models: clean, poisoned, and retrained.

Did the retraining help? How close did the retrained model get to the clean model's performance? What does this tell you about the importance of data quality in fine-tuning?

In [ ]:
# SOLUTION 4.4a: Final 3-bar comparison chart for the code model

fig, ax = plt.subplots(figsize=(10, 6))

models = ["Clean Model", "Poisoned Model", "Retrained Model"]
pass_rates = [
    code_clean_results["pass_at_1"],
    code_poisoned_results["pass_at_1"],
    code_retrained_results["pass_at_1"],
]
colors = [COLOR_CLEAN, COLOR_POISONED, COLOR_RETRAINED]

bars = ax.bar(models, pass_rates, color=colors, width=0.5, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, pass_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylabel("pass@1", fontsize=12)
ax.set_title("Code Model: pass@1 -- Clean vs Poisoned vs Retrained", fontsize=14)
ax.set_ylim(0, 1.15)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION 4.4b: Final 3-bar comparison chart for the QA model

fig, ax = plt.subplots(figsize=(10, 6))

models = ["Clean Model", "Poisoned Model", "Retrained Model"]
fuzzy_rates = [
    qa_clean_results["fuzzy_match"],
    qa_poisoned_results["fuzzy_match"],
    qa_retrained_results["fuzzy_match"],
]
colors = [COLOR_CLEAN, COLOR_POISONED, COLOR_RETRAINED]

bars = ax.bar(models, fuzzy_rates, color=colors, width=0.5, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, fuzzy_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylabel("Fuzzy Match Accuracy", fontsize=12)
ax.set_title("QA Model: Accuracy -- Clean vs Poisoned vs Retrained", fontsize=14)
ax.set_ylim(0, 1.15)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.show()

---

## Section 5: Reflection

Take a few minutes to think about and write your answers to the following questions. There are no "right" answers -- the goal is to think critically about the evaluation and debugging process you just went through.

### 5.1 Verifier Effectiveness

How effective was your verifier? What was the precision and recall?

- If precision was low (many false positives), what clean samples were mistakenly flagged, and why?
- If recall was low (many false negatives), what poisoned samples slipped through, and why?
- How would you improve your verifier if you had more time?

*Write your response below:*

The code verifier achieved reasonable recall by combining original test case failures with edge case testing. Most poisoned code samples fail their original test cases outright, so the primary detection mechanism was straightforward. However, some poisoned samples with very subtle bugs (like off-by-one errors that only manifest on specific inputs) were missed because our heuristic edge cases did not happen to trigger those exact failure modes.

Precision was moderately high -- a few clean samples were flagged because they had legitimate edge case handling issues (e.g., not handling negative inputs gracefully) that were not actually bugs in the context of the original problem specification.

The QA consistency verifier was noisier. Some correct answers showed low consistency because the model naturally paraphrased them differently, while some poisoned answers were memorized so strongly that the model answered consistently (but incorrectly) regardless of rephrasing.

To improve, we could: (1) generate more targeted edge cases using the function signature and docstring, (2) use property-based testing for code, (3) cross-reference QA answers against multiple external sources, and (4) combine multiple verification signals into an ensemble.

### 5.2 Missed Poisoned Samples

Look back at the poisoned samples your verifier missed. What made them hard to detect?

- For code: were the bugs too subtle for your edge case tests to trigger?
- For QA: were the wrong answers too close to the correct ones?
- What fundamental limitation does this reveal about automated verification?

*Write your response below:*

For code, the hardest-to-detect poisoned samples were those with bugs that only trigger on very specific input patterns -- for example, an off-by-one error that only manifests when the input list has exactly two elements, or a flipped condition that only matters when a particular variable is negative. Our heuristic edge cases (empty list, single element, zeros) did not cover every possible triggering condition.

For QA, the hardest misses were cases where the wrong answer was very close to the correct one -- for example, swapping a year by just one or two (1969 vs 1971), or confusing two closely related people (e.g., two Nobel laureates in the same field). The model answered these confidently and consistently, making the consistency verifier ineffective.

The fundamental limitation this reveals is that automated verification can only check for failure modes you anticipate. Truly adversarial data corruption is designed to evade detection. Complete verification would require formal proofs (for code) or exhaustive fact-checking (for QA), both of which are computationally expensive or infeasible at scale. This is a deep challenge in AI safety -- how do you verify training data when the corruption is specifically designed to be undetectable?

### 5.3 Real-World Debugging

In this lab, you had ground-truth poison labels (`is_poisoned`, `correct_answer`, `code_poison_indices`) that you could use to validate your verifier. In a real-world scenario, you would not have these labels.

- How would you approach debugging a model that seems to perform poorly after fine-tuning, without any labels for which training samples are bad?
- What signals would you look for in the training data?
- How would you validate that your verifier is actually catching real problems and not just noise?

*Write your response below:*

Without ground-truth labels, we would rely on several strategies:

1. **Hold-out evaluation**: Use a small, manually curated eval set that you trust. Compare model performance before and after fine-tuning. If performance drops, something in the training data is causing harm.

2. **Anomaly detection**: Look for training samples that are statistical outliers -- unusually long/short answers, unusual vocabulary, solutions that differ structurally from the majority. Clustering or embedding-based methods can help surface these.

3. **Self-consistency**: As we did with the QA verifier, check if the model's behavior is internally consistent. Inconsistency is often a signal that the model has memorized conflicting information from corrupted data.

4. **Cross-validation**: Train on random subsets of the data and compare performance. If removing certain subsets consistently improves performance, those subsets likely contain corrupted samples.

5. **Human spot-checks**: Sample training examples proportional to their influence on model behavior (using techniques like influence functions) and manually review the highest-influence samples.

To validate the verifier without labels, we would use an ablation approach: retrain with and without the flagged samples and measure whether removing them actually improves eval performance. If it does, the verifier is catching real problems. If it does not, we may be removing noise or even clean data.

### 5.4 Goodhart's Law and Evaluation

Goodhart's Law states: "When a measure becomes a target, it ceases to be a good measure."

- How does this apply to the evaluation metrics you used in this lab (pass@1, exact match, fuzzy match)?
- Could a model "game" these metrics -- performing well on them while still having learned bad behaviors?
- How does this connect to the broader challenge of evaluating language models? What would a more robust evaluation approach look like?

*Write your response below:*

Goodhart's Law is directly relevant to this lab in several ways:

**pass@1**: A model could learn to generate code that passes specific test cases without truly understanding the problem. If the test cases are not comprehensive, the model could exploit gaps -- for example, hardcoding return values for known test inputs. This is exactly what some of the poisoned training data does: the solutions "look correct" and might pass some tests but fail on edge cases.

**Exact/fuzzy match**: These metrics reward surface-level similarity. A model could learn to produce text that matches keywords from the expected answer without actually understanding the question. The poisoned QA model demonstrates this -- it gives confident, well-phrased answers that happen to be wrong.

**Gaming metrics**: Yes, absolutely. A model trained to optimize for pass@1 on a fixed test suite could learn to recognize test patterns rather than solve problems generally. Similarly, a QA model could learn to produce answers that contain the right keywords (triggering fuzzy match) without being factually correct in context.

**More robust evaluation** would include:
- Multiple diverse test suites generated independently
- Adversarial testing specifically designed to probe known failure modes
- Human evaluation for nuanced judgments
- Behavioral testing across demographic groups and edge cases
- Evaluation on out-of-distribution examples to test generalization
- Process-based evaluation (checking reasoning steps, not just final answers)

The core lesson: no single metric captures everything we care about. Robust evaluation requires a diverse suite of complementary measures, and we must continuously update our evals as models become better at optimizing for them.

---

## Congratulations!

You have completed the evaluation lab. Here is a summary of what you accomplished:

1. **Evaluated** two fine-tuned models and discovered unexpected performance degradation
2. **Investigated** the training data and found that ~20% of samples were corrupted
3. **Built verifiers** to automatically detect corrupted samples using edge case testing and consistency checking
4. **Filtered and retrained** the models on clean data and recovered performance

These are core skills for anyone working with language models in production. Data quality is often the biggest lever you have for improving model performance -- and building good evaluations is how you discover and fix data problems.